# Практическая работа №2. Часть 1О
## Градиентные методы оптимизации

**Цель:** написать градиентный спуск, импульс и Adam «с нуля» и увидеть на простой функции, как скорость обучения и выбор метода меняют путь к минимуму; написать свой оптимизатор как наследник `torch.optim.Optimizer` и сравнить готовые оптимизаторы на настоящей сверточной сети.

> **Правило части.** В заданиях О1–О4 оптимизаторы пишутся на тензорах, без `torch.optim`. В О5 – свой класс-наследник `torch.optim.Optimizer`, готовый `torch.optim.SGD` – только для сверки. В О6 используются готовые `torch.optim` и планировщики.

**Задача:** сквозной пример со слайдов – функция двух переменных $f(w_1, w_2) = w_1^2 + 10\,w_2^2$ со стартом в точке $(-4, 1)$; спуск останавливается, когда точка подошла к минимуму $(0, 0)$ ближе чем на $0{,}01$ ($\lVert w \rVert < 0{,}01$). В конце – обучение сети `SimpleCNN` из части 4А на FashionMNIST разными оптимизаторами.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается»; для обучения сети указаны качественные ожидания. Сначала прочитайте теорию ниже.

## Как устроены оптимизаторы

**1. Градиентный спуск.** Обучение – это поиск весов, при которых потеря минимальна. **Градиент** $\nabla f$ – вектор частных производных – показывает направление самого быстрого роста функции, значит, против него функция убывает быстрее всего (Гудфеллоу, раздел 4.3). Шаг спуска:

$$w \leftarrow w - \eta\,\nabla f(w),$$

где $\eta$ – **скорость обучения** (learning rate). Для нашей функции $\nabla f = (2w_1,\ 20w_2)$. Первый шаг из $(-4, 1)$ при $\eta = 0{,}05$: градиент $(-8, 20)$, сдвиг $\eta\nabla f = (-0{,}4;\ 1)$, новая точка $(-4 + 0{,}4;\ 1 - 1) = (-3{,}6;\ 0)$.

**2. Линии уровня и «овраг».** Линии, на которых $f$ постоянна, – вытянутые эллипсы: по $w_2$ функция в 10 раз круче, чем по $w_1$. Это «овраг» с крутыми стенками и пологим дном. Шаг спуска действует на каждую координату отдельно:

$$w_1 \leftarrow (1 - 2\eta)\,w_1, \qquad w_2 \leftarrow (1 - 20\eta)\,w_2.$$

- Спуск сходится, только если оба множителя по модулю меньше 1. Для второй координаты $|1 - 20\eta| < 1$ дает $\eta < 0{,}1$: при $\eta = 0{,}11$ множитель $1 - 2{,}2 = -1{,}2$, точка прыгает со стенки на стенку все выше – **расходимость**.
- Если $1 - 20\eta < 0$ (то есть $\eta > 0{,}05$), знак $w_2$ меняется каждый шаг – путь идет **зигзагом** поперек оврага.
- Вдоль дна множитель $1 - 2\eta$ близок к 1, и точка ползет медленно: при $\eta = 0{,}01$ он равен $0{,}98$, и от $w_1 = -4$ до $|w_1| < 0{,}01$ нужно около $\ln 400 / 0{,}02 \approx 300$ шагов.

Лучший постоянный шаг уравнивает оба множителя по модулю: $\eta^* = \dfrac{2}{2 + 20} \approx 0{,}09$. Трудность в том, что крутизна по разным направлениям разная (отношение $20 / 2 = 10$ называют **числом обусловленности**; раздел 8.2.1). В нейросетях направлений миллионы, и их крутизна различается в тысячи раз.

**3. Стохастичность и мини-пакеты.** Потеря сети – среднее по всем $n$ обучающим объектам $L = \frac1n \sum_i \ell_i$, и точный градиент требует прогнать все объекты. **Стохастический градиентный спуск** (SGD) берет случайный **мини-пакет** из $B$ объектов и шагает по его градиенту $\frac1B \sum_{i \in \text{пакет}} \nabla \ell_i$ – это несмещенная оценка полного градиента, «шум» которой уменьшается как $1/\sqrt{B}$ (раздел 8.1.3): пакет из 64 объектов «шумит» в 8 раз меньше одного объекта. Термины: **итерация** – один шаг по одному пакету, **эпоха** – проход по всем объектам ($60\,000 / 64 \approx 938$ итераций для FashionMNIST). Шум мешает точно сесть в минимум (поэтому к концу обучения шаг уменьшают), зато помогает выбираться из плоских областей и седловых точек.

**4. Импульс.** Представьте тяжелый шар, который катится по оврагу: он набирает скорость вдоль дна, а поперечные толчки гасятся инерцией. Метод импульса (momentum, раздел 8.3.2) хранит «скорость» $v$ – накопленную сумму градиентов:

$$v \leftarrow \beta\,v + \nabla f(w), \qquad w \leftarrow w - \eta\,v, \qquad v_0 = 0.$$

Если градиент долго смотрит в одну сторону, шаг вырастает до $\eta / (1 - \beta)$ – при $\beta = 0{,}9$ в 10 раз; если градиент меняет знак (зигзаг), вклады взаимно гасятся. Вариант **Нестерова** (раздел 8.3.3) считает градиент в точке, куда шар «собирается» попасть, – это чуть точнее. Такую же форму обновления использует `torch.optim.SGD(..., momentum=β)`.

*Пример.* $\eta = 0{,}05$, $\beta = 0{,}5$. Шаг 1 совпадает с обычным: $v = (-8, 20)$, точка $(-3{,}6;\ 0)$. Шаг 2: градиент $(-7{,}2;\ 0)$, $v = 0{,}5 \cdot (-8, 20) + (-7{,}2;\ 0) = (-11{,}2;\ 10)$, точка $(-3{,}6 + 0{,}56;\ 0 - 0{,}5) = (-3{,}04;\ -0{,}5)$. До цели – 17 шагов вместо 57.

**5. Адаптивные методы: у каждого веса своя скорость.** Идея – делить шаг каждого веса на «типичную величину» его градиента, чтобы крутые направления шагали осторожно, а пологие – смело (раздел 8.5):
- **AdaGrad**: копит сумму квадратов градиентов $r \leftarrow r + g^2$ и шагает $w \leftarrow w - \eta\,g / (\sqrt{r} + \varepsilon)$ (все операции поэлементные). Минус: $r$ только растет, и шаг со временем угасает;
- **RMSProp**: заменяет сумму скользящим средним $r \leftarrow \rho\,r + (1 - \rho)\,g^2$ – «память» о старых градиентах забывается;
- **Adam** (раздел 8.5.3) соединяет RMSProp с импульсом. На шаге $t$:

$$m \leftarrow \beta_1 m + (1 - \beta_1)\,g, \qquad v \leftarrow \beta_2 v + (1 - \beta_2)\,g^2,$$
$$\hat m = \frac{m}{1 - \beta_1^t}, \qquad \hat v = \frac{v}{1 - \beta_2^t}, \qquad w \leftarrow w - \eta\,\frac{\hat m}{\sqrt{\hat v} + \varepsilon}.$$

Обычно $\beta_1 = 0{,}9$, $\beta_2 = 0{,}999$, $\varepsilon = 10^{-8}$. Деление на $1 - \beta^t$ – **поправка смещения**: $m$ и $v$ стартуют с нуля, и без поправки первые шаги были бы слишком малы. На первом шаге $\hat m = g$, $\hat v = g^2$, и шаг равен $\eta \cdot g / |g| = \eta$ по каждой координате, независимо от крутизны: из $(-4, 1)$ при $\eta = 0{,}1$ получается $(-3{,}9;\ 0{,}9)$.

**6. Нет лучшего метода на все случаи.** На нашей простой функции Adam с $\eta = 0{,}1$ доходит до цели за 105 шагов – дольше импульса (17): у квадратичной функции размер шага Adam не зависит от величины градиента, и у самого минимума он «топчется». На нейросетях Adam обычно быстрее всего снижает потерю и мало требует от выбора $\eta$, а SGD с импульсом и хорошим расписанием часто дает чуть лучшую точность на тесте (раздел 8.5.4). Поэтому метод и $\eta$ подбирают по опыту на своей задаче.

**7. Планировщики скорости обучения.** Скорость обучения меняют по ходу обучения:
- **ступенчатое** уменьшение – например, в 10 раз каждые $k$ эпох;
- **косинусное**: $\eta_t = \eta_{\max} \cdot \frac12\big(1 + \cos(\pi\,t / T)\big)$ – плавно от $\eta_{\max}$ до 0 за $T$ шагов;
- **разогрев** (warmup): первые шаги $\eta$ линейно растет от малого значения до $\eta_{\max}$, пока статистики Adam и веса сети не «устоялись».

**8. Оптимизатор в PyTorch.** Класс `torch.optim.Optimizer` хранит **группы параметров** `param_groups` – списки тензоров со своими настройками (`lr`, `momentum`) – и **состояние** `state[p]` для каждого параметра (например, скорость $v$ у импульса или $m$, $v$ у Adam). Метод `step()` проходит по всем параметрам и меняет их на месте по градиенту `p.grad`; он выполняется внутри `torch.no_grad()`, потому что обновление весов – не часть графа вычислений. `zero_grad()` уже написан в базовом классе. Планировщик (`torch.optim.lr_scheduler`) после каждого шага меняет `lr` в группах.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
from pathlib import Path
import math
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# FashionMNIST для О5–О6 заранее скачивается в папку data (интернет в аудитории не нужен):
#   datasets.FashionMNIST('data', train=True, download=True); datasets.FashionMNIST('data', train=False, download=True)
DATA_DIR = Path('data')
tf = transforms.ToTensor()
try:
    train_ds = datasets.FashionMNIST(DATA_DIR, train=True, download=False, transform=tf)
    test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=False, transform=tf)
except Exception:
    print('FashionMNIST не найден в папке data – используются случайные картинки FakeData')
    train_ds = datasets.FakeData(size=10000, image_size=(1, 28, 28), num_classes=10, transform=tf)
    test_ds = datasets.FakeData(size=2000, image_size=(1, 28, 28), num_classes=10, transform=tf, random_offset=10000)
small_train = Subset(train_ds, range(10000))    # 10 000 картинок: одна эпоха – 157 итераций
small_test = Subset(test_ds, range(2000))
test_loader = DataLoader(small_test, batch_size=500)

# Сеть и функции из части 4А ПР1
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

def accuracy(model, loader):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(dim=1) == y).sum().item()
            n += len(y)
    return correct / n

In [ ]:
# Учебный пример со слайдов: f(w1, w2) = w1^2 + 10 * w2^2
def f(w):
    return w[0] ** 2 + 10 * w[1] ** 2

w_start = torch.tensor([-4.0, 1.0], dtype=torch.float64)   # float64 – чтобы число шагов не зависело от округлений
TOL = 0.01                                                  # остановка: ||w|| < 0.01
print(show(f(w_start)), '  ожидается 26.0 (16 + 10)')

**Задание О1. Градиент и путь градиентного спуска.**

- `grad_f(w)` – градиент нашей функции $\nabla f = (2w_1,\ 20w_2)$ в виде тензора из двух чисел (`torch.stack` из двух выражений сохранит тип `float64`).
- `sgd_path(w0, lr, max_steps=5000, tol=TOL)` – путь спуска: список точек, начиная с `w0`. На каждом шаге $w \leftarrow w - \eta\,\nabla f(w)$ и новая точка добавляется в список. Остановиться, когда $\lVert w \rVert < $ `tol` (дошли) или $\lVert w \rVert > 10^6$ (разошлись). Норма – `torch.linalg.norm(w)`. Вернуть `torch.stack(path)` – тензор формы $(k + 1, 2)$, где $k$ – число сделанных шагов.

Не меняйте `w0` на месте: пишите `w = w - lr * g`, а не `w -= lr * g`, иначе в списке окажутся ссылки на один и тот же тензор.

*Проверка по п. 1–2 теории.* Первый шаг при $\eta = 0{,}05$ – в точку $(-3{,}6;\ 0)$. Число шагов: $\eta = 0{,}01$ – 297, $0{,}05$ – 57, $0{,}09$ – 31 (почти лучший шаг $\eta^* \approx 0{,}09$), $0{,}11$ – расходится.

In [ ]:
def grad_f(w):
    return ...                                  # <-- (2 w1, 20 w2), torch.stack

def sgd_path(w0, lr, max_steps=5000, tol=TOL):
    w, path = w0, [w0]
    for step in range(max_steps):
        w = ...                                 # <-- шаг против градиента
        path.append(w)
        if ...:                                 # <-- дошли или разошлись: норма < tol или > 1e6
            break
    return torch.stack(path)


print(show(grad_f(w_start)), '  ожидается [-8.0, 20.0]')
p05 = sgd_path(w_start, 0.05)
print(show(p05[1]), '  ожидается [-3.6, 0.0]')
for lr in [0.01, 0.05, 0.09]:
    print(lr, len(sgd_path(w_start, lr)) - 1, '  ожидается', {0.01: 297, 0.05: 57, 0.09: 31}[lr])
p11 = sgd_path(w_start, 0.11)
print(torch.linalg.norm(p11[-1]).item() > 1e6, '  ожидается True (при lr = 0.11 спуск расходится)')

**Вопрос О1.** а) Почему при $\eta = 0{,}05$ вторая координата после первого же шага стала ровно нулем? б) Почему при $\eta = 0{,}11$ спуск расходится, хотя по первой координате шаг совсем небольшой? в) Сравните с градиентным спуском для логистической регрессии в 9 классе: что там играло роль $\eta$?

*Ваш ответ:*

**Задание О2. Множители и траектории.**

Функция `factors(lr)` возвращает тензор двух множителей из п. 2 теории: $(1 - 2\eta,\ 1 - 20\eta)$ – во сколько раз каждый шаг уменьшает $w_1$ и $w_2$. По ним можно предсказать поведение спуска без запуска: модуль меньше 1 – координата сходится, отрицательный множитель – зигзаг, модуль больше 1 – расходимость.

Готовая ячейка рисует пути для четырех $\eta$ на линиях уровня $f$.

In [ ]:
def factors(lr):
    return ...          # <-- тензор (1 - 2 lr, 1 - 20 lr)


for lr in [0.01, 0.05, 0.09, 0.11]:
    print(lr, show(factors(lr)))
print('  ожидается [0.98, 0.8], [0.9, 0.0], [0.82, -0.8], [0.78, -1.2]')

In [ ]:
# Готовая ячейка: траектории на линиях уровня
g1, g2 = np.meshgrid(np.linspace(-4.5, 1.0, 300), np.linspace(-1.6, 1.6, 300))
fig, axs = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
for ax, lr in zip(axs, [0.01, 0.05, 0.09, 0.11]):
    ax.contour(g1, g2, g1 ** 2 + 10 * g2 ** 2, levels=[0.25, 1, 4, 9, 16, 26], colors='#9A9AA3', linewidths=0.8)
    p = sgd_path(w_start, lr, max_steps=40 if lr == 0.11 else 5000).numpy()
    p = p[np.abs(p[:, 1]) < 1.6][:60]
    ax.plot(p[:, 0], p[:, 1], '-o', color='#8F8260', ms=2.5, lw=1.2)
    ax.plot(0, 0, '*', color='#1C1C22', ms=10)
    n_steps = len(sgd_path(w_start, lr)) - 1
    ax.set_title(f'lr = {lr}: ' + ('расходится' if lr == 0.11 else f'{n_steps} шагов'), fontsize=10)
    ax.set_xlim(-4.5, 1); ax.set_ylim(-1.6, 1.6)
fig.tight_layout(); fig.savefig('sgd_paths.png', dpi=110); plt.close(fig)
display(Image('sgd_paths.png'))

**Вопрос О2.** а) По множителям объясните, почему при $\eta = 0{,}09$ путь идет зигзагом, но сходится быстрее всех. б) Что станет с числом шагов, если функцию заменить на $w_1^2 + 100\,w_2^2$? Какой будет наибольший допустимый $\eta$?

*Ваш ответ:*

**Задание О3. Метод импульса.**

`momentum_path(w0, lr, beta, max_steps=5000, tol=TOL)` – как `sgd_path`, но с «скоростью» $v$ (п. 4 теории): $v_0 = 0$, на каждом шаге $v \leftarrow \beta v + \nabla f(w)$, затем $w \leftarrow w - \eta v$. Условия остановки те же.

*Проверка.* При $\eta = 0{,}05$, $\beta = 0{,}5$: первая точка $(-3{,}6;\ 0)$, вторая $(-3{,}04;\ -0{,}5)$, до цели 17 шагов. При $\beta = 0$ метод превращается в обычный спуск – 57 шагов.

In [ ]:
def momentum_path(w0, lr, beta, max_steps=5000, tol=TOL):
    w, v, path = w0, torch.zeros_like(w0), [w0]
    for step in range(max_steps):
        v = ...                                 # <-- beta * v + градиент
        w = ...                                 # <-- шаг по скорости
        path.append(w)
        if torch.linalg.norm(w) < tol or torch.linalg.norm(w) > 1e6:
            break
    return torch.stack(path)


pm = momentum_path(w_start, 0.05, 0.5)
print(show(pm[1]), show(pm[2]), '  ожидается [-3.6, 0.0] [-3.04, -0.5]')
print(len(pm) - 1, len(momentum_path(w_start, 0.05, 0.0)) - 1, '  ожидается 17 57')

**Вопрос О3.** а) Почему на втором шаге импульс увел $w_2$ из нуля в $-0{,}5$, хотя градиент по $w_2$ там был равен нулю? Хорошо это или плохо? б) Попробуйте $\beta = 0{,}9$ при том же $\eta$. Сколько шагов? Объясните через «шар с инерцией».

*Ваш ответ:*

**Задание О4. Adam.**

`adam_path(w0, lr, beta1=0.9, beta2=0.999, eps=1e-8, max_steps=5000, tol=TOL)` – по формулам из п. 5 теории (Гудфеллоу, раздел 8.5.3). Номер шага $t$ начинается с 1 – это важно для поправки смещения. Все операции поэлементные: `g ** 2`, `torch.sqrt(v_hat)`.

*Проверка.* При $\eta = 0{,}1$ первый шаг смещает каждую координату ровно на $0{,}1$: $(-3{,}9;\ 0{,}9)$ – хотя градиент по $w_2$ в 2,5 раза больше, чем по $w_1$. До цели – 105 шагов (п. 6 теории).

In [ ]:
def adam_path(w0, lr, beta1=0.9, beta2=0.999, eps=1e-8, max_steps=5000, tol=TOL):
    w, path = w0, [w0]
    m, v = torch.zeros_like(w0), torch.zeros_like(w0)
    for t in range(1, max_steps + 1):
        g = grad_f(w)
        m = ...                                 # <-- скользящее среднее градиента
        v = ...                                 # <-- скользящее среднее квадрата градиента
        m_hat = ...                             # <-- поправка смещения для m
        v_hat = ...                             # <-- поправка смещения для v
        w = ...                                 # <-- шаг Adam
        path.append(w)
        if torch.linalg.norm(w) < tol or torch.linalg.norm(w) > 1e6:
            break
    return torch.stack(path)


pa = adam_path(w_start, 0.1)
print(show(pa[1]), '  ожидается [-3.9, 0.9]')
print(len(pa) - 1, len(adam_path(w_start, 0.3)) - 1, '  ожидается 105 61')

In [ ]:
# Готовая ячейка: три метода на одной картинке
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.contour(g1, g2, g1 ** 2 + 10 * g2 ** 2, levels=[0.25, 1, 4, 9, 16, 26], colors='#9A9AA3', linewidths=0.8)
for p, c, lab in [(sgd_path(w_start, 0.05), '#9A9AA3', 'SGD, lr 0.05'),
                  (momentum_path(w_start, 0.05, 0.5), '#8F8260', 'импульс 0.5, lr 0.05'),
                  (adam_path(w_start, 0.1), '#1C1C22', 'Adam, lr 0.1')]:
    p = p.numpy()
    ax.plot(p[:, 0], p[:, 1], '-o', color=c, ms=2, lw=1.3, label=f'{lab}: {len(p) - 1} шагов')
ax.plot(0, 0, '*', color='#B8AB84', ms=12); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig('three_methods.png', dpi=110); plt.close(fig)
display(Image('three_methods.png'))

**Вопрос О4.** а) Почему первый шаг Adam одинаков по обеим координатам? б) Почему на этой функции Adam медленнее импульса, а в нейросетях его выбирают чаще всего?

*Ваш ответ:*

**Задание О5. Свой оптимизатор `MySGD`.**

Напишите класс `MySGD(params, lr, momentum)` – наследник `torch.optim.Optimizer` (п. 8 теории), который делает то же, что `torch.optim.SGD`:
- в `__init__` передать в базовый класс параметры и словарь настроек по умолчанию `dict(lr=lr, momentum=momentum)`;
- метод `step()` помечен `@torch.no_grad()`. Для каждой группы из `self.param_groups` и каждого параметра `p` с градиентом:
  - если `momentum` не 0: взять состояние `self.state[p]` (словарь, который базовый класс заводит для каждого параметра); на первом шаге сохранить в нем скорость `v` как **копию** градиента `p.grad.clone()`, на следующих – обновить на месте `v.mul_(momentum).add_(p.grad)`, то есть $v \leftarrow \beta v + g$; шагать по `v`;
  - обновить параметр на месте: `p.add_(шаг, alpha=-lr)` означает $p \leftarrow p - \eta \cdot \text{шаг}$.

На первом шаге PyTorch берет $v = g$ – это то же, что $v = \beta \cdot 0 + g$ из О3. Методы с подчеркиванием (`mul_`, `add_`) меняют тензор на месте – состояние и веса должны обновляться именно так.

*Сверка.* Две одинаковые сети `SimpleCNN` из части 4А (один сид) делают по 5 шагов на одном и том же пакете: одна – своим `MySGD`, другая – `torch.optim.SGD`. Веса должны совпасть (расхождение порядка $10^{-7}$ и меньше – из-за порядка операций в `float32`).

In [ ]:
class MySGD(torch.optim.Optimizer):
    def __init__(self, params, lr=0.01, momentum=0.0):
        super().__init__(params, ...)                     # <-- словарь настроек по умолчанию

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, mom = group['lr'], group['momentum']
            for p in group['params']:
                if p.grad is None:
                    continue
                d = p.grad
                if mom != 0:
                    state = self.state[p]
                    if 'v' not in state:
                        state['v'] = ...                  # <-- копия градиента
                    else:
                        ...                               # <-- v = mom * v + grad, на месте
                    d = state['v']
                ...                                       # <-- p = p - lr * d, на месте


def compare_with_torch(momentum, steps=5):
    g = torch.Generator().manual_seed(RANDOM_STATE)
    x, y = torch.randn(32, 1, 28, 28, generator=g), torch.randint(0, 10, (32,), generator=g)
    nets = []
    for make_opt in [lambda ps: MySGD(ps, lr=0.05, momentum=momentum),
                     lambda ps: torch.optim.SGD(ps, lr=0.05, momentum=momentum)]:
        torch.manual_seed(RANDOM_STATE)
        net = SimpleCNN()
        opt = make_opt(net.parameters())
        for _ in range(steps):
            opt.zero_grad()
            F.cross_entropy(net(x), y).backward()
            opt.step()
        nets.append(net)
    return max((a - b).abs().max().item() for a, b in zip(nets[0].parameters(), nets[1].parameters()))


for mom in [0.0, 0.9]:
    print(f'momentum={mom}: {compare_with_torch(mom):.1e}', '  ожидается < 1e-06')

**Вопрос О5.** а) Зачем на первом шаге сохранять `p.grad.clone()`, а не сам `p.grad`? б) Зачем состояние хранится отдельно для каждого параметра? Сколько дополнительной памяти требуют SGD с импульсом и Adam по сравнению с самой сетью?

*Ваш ответ:*

**Задание О6. Оптимизаторы и расписание на настоящей сети.**

- `make_optimizer(name, params, lr)` возвращает готовый оптимизатор: `'sgd'` – `torch.optim.SGD` без импульса, `'momentum'` – SGD с `momentum=0.9`, `'adam'` – `torch.optim.Adam`. Для любого другого имени – `raise ValueError(name)`.
- `cosine_lr(step, total, lr_max, warmup=0)` – скорость обучения на шаге `step` (п. 7 теории): пока `step < warmup` – линейный разогрев $\eta_{\max} \cdot (\text{step} + 1) / \text{warmup}$; дальше – косинус от $\eta_{\max}$ до 0 по доле пройденного пути $\text{progress} = (\text{step} - \text{warmup}) / (\text{total} - \text{warmup})$: $\eta_{\max} \cdot \frac12 (1 + \cos(\pi \cdot \text{progress}))$.

Готовые ячейки: график расписания; сравнение SGD, SGD с импульсом и Adam (у Adam – с косинусным расписанием и без) на одной эпохе по 10 000 картинок FashionMNIST; подбор $\eta$ для обычного SGD по сетке $\{0{,}001;\ 0{,}01;\ 0{,}1;\ 1\}$. Все запуски вместе – около минуты.

In [ ]:
def make_optimizer(name, params, lr):
    if name == 'sgd':
        return ...                                # <-- SGD без импульса
    if name == 'momentum':
        return ...                                # <-- SGD с momentum=0.9
    if name == 'adam':
        return ...                                # <-- Adam
    raise ValueError(name)

def cosine_lr(step, total, lr_max, warmup=0):
    if step < warmup:
        return ...                                # <-- линейный разогрев
    progress = (step - warmup) / (total - warmup)
    return ...                                    # <-- косинусное затухание


opt = make_optimizer('momentum', SimpleCNN().parameters(), 0.01)
print(type(opt).__name__, opt.defaults['momentum'], type(make_optimizer('adam', SimpleCNN().parameters(), 1e-3)).__name__,
      '  ожидается SGD 0.9 Adam')
print([round(cosine_lr(s, 100, 0.1, warmup=10), 4) for s in [0, 9, 10, 55, 100]],
      '  ожидается [0.01, 0.1, 0.1, 0.05, 0.0]')

In [ ]:
# Готовая ячейка: как выглядит расписание (157 итераций, разогрев 16)
TOTAL = 157
steps = np.arange(TOTAL)
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(steps, [cosine_lr(s, TOTAL, 3e-3, warmup=16) for s in steps], color='#8F8260', lw=2)
ax.set_xlabel('итерация'); ax.set_ylabel('lr'); ax.set_title('Разогрев 16 итераций + косинус')
fig.tight_layout(); fig.savefig('cosine_schedule.png', dpi=110); plt.close(fig)
display(Image('cosine_schedule.png'))

In [ ]:
# Готовая ячейка: одна эпоха на 10 000 картинок для каждого варианта
def run(name, lr, schedule=False):
    torch.manual_seed(RANDOM_STATE)
    model = SimpleCNN()
    opt = make_optimizer(name, model.parameters(), lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: cosine_lr(s, TOTAL, 1.0, warmup=16)) if schedule else None
    loader = DataLoader(small_train, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(RANDOM_STATE))
    model.train()
    hist = []
    for x, y in loader:
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        if sched is not None:
            sched.step()
        hist.append(loss.item())
    return hist, accuracy(model, test_loader)

variants = [('SGD, lr 0.01', 'sgd', 0.01, False), ('импульс 0.9, lr 0.01', 'momentum', 0.01, False),
            ('Adam, lr 0.001', 'adam', 1e-3, False), ('Adam, lr 0.003 + косинус', 'adam', 3e-3, True)]
res = {}
start = time.perf_counter()
for label, name, lr, sch in variants:
    res[label] = run(name, lr, sch)
    print(f'{label:<26} потеря в конце {np.mean(res[label][0][-20:]):.3f}   точность на тесте {res[label][1]:.3f}')
print(f'время: {time.perf_counter() - start:.0f} с')
end_loss = {k: np.mean(v[0][-20:]) for k, v in res.items()}
print(end_loss['SGD, lr 0.01'] > max(end_loss['импульс 0.9, lr 0.01'], end_loss['Adam, lr 0.001']),
      '  ожидается True (обычный SGD с малым шагом отстает)')

fig, ax = plt.subplots(figsize=(7, 3.4))
for (label, (h, _)), c in zip(res.items(), ['#9A9AA3', '#8F8260', '#1C1C22', '#B8AB84']):
    ax.plot(np.convolve(h, np.ones(10) / 10, mode='valid'), color=c, lw=2, label=label)
ax.set_xlabel('итерация'); ax.set_ylabel('потеря'); ax.legend(fontsize=8); ax.set_title('Одна эпоха, 10 000 картинок')
fig.tight_layout(); fig.savefig('optimizers_compare.png', dpi=110); plt.close(fig)
display(Image('optimizers_compare.png'))

In [ ]:
# Готовая ячейка: подбор lr для обычного SGD по сетке
for lr in [0.001, 0.01, 0.1, 1.0]:
    h, acc_t = run('sgd', lr)
    print(f'lr = {lr:<6} потеря в конце {np.mean(h[-20:]):.3f}   точность {acc_t:.3f}')
print('  ожидается: при 0.001 обучение едва началось, при 1.0 – разваливается (nan или около 2.3), лучший – в середине сетки')

**Вопрос О6.** а) Какой вариант быстрее всех снизил потерю? Совпадает ли порядок по потере с порядком по точности на тесте? б) Объясните результаты сетки по $\eta$ через п. 2 теории: что общего у $\eta = 1$ на сети и $\eta = 0{,}11$ на функции $f$? в) Зачем Adam с большим шагом понадобился разогрев?

*Ваш ответ:*

### Задание повышенной сложности (к части 1О)

1. **Функция Розенброка.** Запустите свои `sgd_path`, `momentum_path` и `adam_path` на $f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$ со старта $(-1{,}5;\ 2)$ (минимум в $(1, 1)$). Подберите для каждого метода шаг, при котором он доходит до минимума быстрее всего, и нарисуйте пути на линиях уровня.
2. **Поиск скорости обучения (LR range test).** Обучайте `SimpleCNN` одну эпоху, увеличивая $\eta$ после каждой итерации в геометрической прогрессии от $10^{-5}$ до $1$. Постройте график потери от $\eta$ в логарифмическом масштабе и выберите $\eta$ чуть левее места, где потеря начинает расти.
3. **Размер пакета.** Обучите `SimpleCNN` одну эпоху с пакетами 8, 64 и 512 при одинаковом $\eta$. Сравните время, число итераций и точность. Как изменится результат, если $\eta$ увеличивать пропорционально размеру пакета?
4. **Нестеров.** Добавьте в `MySGD` параметр `nesterov` и сверьте с `torch.optim.SGD(..., nesterov=True)`.

In [ ]:
# Место для задания повышенной сложности